# Streamlit 앱 Colab 테스트

## Goal

GitHub의 최신 `streamlit_app`을 Colab에 내려받아 의존성, 단위 테스트, 전처리 함수, Streamlit 화면을 순서대로 검증합니다. 기존 Colab 분석 프로그램은 수정하지 않습니다.

> 실행 전 로컬 변경사항을 GitHub `main` 브랜치에 push해야 합니다. 공개 Google Drive 파일 다운로드 기능 자체는 실제 공개 링크가 있을 때 앱 화면에서 확인합니다.

## Setup

### 1. 저장소 준비

아래 설정을 바꾸면 다른 브랜치도 테스트할 수 있습니다. 셀은 기존 clone을 삭제하지 않고 fetch/reset하여 재실행할 수 있게 구성했습니다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

REPO_URL = "https://github.com/JongdaeChoi/Exhibition-Data-Analysis.git"
BRANCH = "main"
REPO_DIR = Path("/content/Exhibition-Data-Analysis")
APP_DIR = REPO_DIR / "streamlit_app"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

required_files = [APP_DIR / "app.py", APP_DIR / "requirements-dev.txt"]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("GitHub에 Streamlit 코드가 없습니다. 로컬 변경을 push한 뒤 다시 실행하세요: " + ", ".join(missing))

os.chdir(APP_DIR)
print("테스트 대상:", APP_DIR)
print("커밋:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

### 2. 의존성 설치

Streamlit 실행 및 테스트에 필요한 패키지를 앱 전용 선언 파일에서 설치합니다.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dev.txt"],
    check=True,
)
print("의존성 설치 완료")

## Steps

### 3. 전체 단위 테스트

파일 적재, 원본/작업본 분리, 기본현황, 결측값, 특정값, Column 삭제, 날짜 분리 테스트를 실행합니다.

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("단위 테스트가 실패했습니다. 위 오류를 확인하세요.")

### 4. 전처리 핵심 동작 확인

작은 샘플로 `df` 불변성, 결측값 대체, 특정값 변경, 날짜 요소 분리를 눈으로 확인합니다. 같은 Colab 런타임에서 재실행해도 최신 전처리 코드를 사용하도록 모듈을 다시 불러옵니다.

In [ ]:
import importlib
import pandas as pd
import data.preprocessing as preprocessing_module
importlib.reload(preprocessing_module)

from data.preprocessing import (
    comparison_summary,
    fill_missing_values,
    replace_selected_value,
    split_date_components,
)

df = pd.DataFrame({
    "지역": ["서울", " 서울 " , None],
    "매출": [100.0, None, 300.0],
    "등록일시": ["2026-07-15 09:30:45", "2026-07-16 17:00:00", None],
})
df_clean = df.copy(deep=True)
df_clean = fill_missing_values(df_clean, "매출", "평균값").frame
df_clean = replace_selected_value(df_clean, "지역", " 서울 ", "서울").frame
df_clean = split_date_components(df_clean, "등록일시", ["year_month_day", "hour"]).frame

assert pd.isna(df.loc[1, "매출"]), "원본 df가 변경되었습니다."
display(df_clean)
display(comparison_summary(df, df_clean))
print("원본 df 불변성 및 전처리 핵심 동작 확인 완료")

### 5. Streamlit 앱 실행

백그라운드에서 앱을 실행하고 `/healthz` 응답을 확인합니다.

In [ ]:
import requests
from google.colab import userdata

PORT = 8501
if "streamlit_process" in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)

streamlit_environment = os.environ.copy()
try:
    gemini_api_key = userdata.get("exhibition")
except Exception as exc:
    gemini_api_key = None
    print("참고: Colab Secret exhibition을 읽지 못했습니다. 앱 화면에서 API 키를 입력할 수 있습니다.", exc)
if gemini_api_key:
    streamlit_environment["GEMINI_API_KEY"] = gemini_api_key
    print("Colab Secret exhibition을 Streamlit 프로세스에 전달했습니다.")

streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.headless=true", f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--server.enableWebsocketCompression=false",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    env=streamlit_environment,
)

health_url = f"http://127.0.0.1:{PORT}/_stcore/health"
for _ in range(30):
    try:
        response = requests.get(health_url, timeout=2)
        if response.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    streamlit_process.terminate()
    raise RuntimeError("Streamlit 서버가 30초 안에 시작되지 않았습니다.")

print("Streamlit health check:", response.status_code, response.text.strip())

### 6. 임시 HTTPS 주소로 앱 열기

Colab의 `proxyPort`와 kernel-port iframe은 런타임에 따라 404 또는 빈 화면을 반환할 수 있습니다. 아래 셀은 Cloudflare Quick Tunnel로 임시 HTTPS 주소를 생성합니다. Cloudflare 계정이나 토큰은 필요하지 않습니다. **이 주소는 공개 주소이므로 테스트 중 민감하거나 개인정보가 포함된 파일을 업로드하지 마세요.** Colab 런타임이 종료되면 앱과 터널도 함께 종료됩니다.

In [ ]:
import re
from IPython.display import HTML, display

if streamlit_process.poll() is not None:
    raise RuntimeError("Streamlit 서버가 종료되었습니다. 바로 위의 앱 실행 셀부터 다시 실행하세요.")

cloudflared_path = Path("/content/cloudflared")
if not cloudflared_path.exists():
    download_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    binary_response = requests.get(download_url, timeout=120)
    binary_response.raise_for_status()
    cloudflared_path.write_bytes(binary_response.content)
    cloudflared_path.chmod(0o755)

if "cloudflared_process" in globals() and cloudflared_process.poll() is None:
    cloudflared_process.terminate()
    cloudflared_process.wait(timeout=10)

tunnel_log_path = Path("/content/cloudflared.log")
tunnel_log_handle = tunnel_log_path.open("w", encoding="utf-8")
cloudflared_process = subprocess.Popen(
    [str(cloudflared_path), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)

tunnel_url = None
for _ in range(60):
    time.sleep(1)
    log_text = tunnel_log_path.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        tunnel_url = match.group(0)
        break
    if cloudflared_process.poll() is not None:
        break

if not tunnel_url:
    raise RuntimeError("Cloudflare 임시 주소 생성 실패:\n" + tunnel_log_path.read_text(encoding="utf-8", errors="replace")[-4000:])

display(HTML(f'<a href="{tunnel_url}" target="_blank" style="font-size:20px;font-weight:bold">Streamlit 앱 새 창에서 열기</a>'))
print("임시 공개 주소:", tunnel_url)

# Colab 런타임에서 공개 URL로 되돌아가는 요청은 차단될 수 있으므로 참고용으로만 확인합니다.
tunnel_check_status = None
for _ in range(5):
    try:
        tunnel_response = requests.get(tunnel_url, timeout=5)
        tunnel_check_status = tunnel_response.status_code
        if tunnel_check_status == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if tunnel_check_status == 200:
    print("터널 확인 완료: HTTP 200")
else:
    print("참고: Colab 내부 재조회는 확인되지 않았지만 위 링크는 정상 생성되었습니다. 브라우저에서 링크를 여세요.")

## Checks

앱에서 다음 항목을 직접 확인하세요.

1. 로컬 CSV/XLSX 파일 선택 및 적재 진행 표시
2. `df`와 `df_clean` 분리 안내 및 기본현황 표
3. 결측값 테이블에서 처리값만 입력해도 처리버튼 활성화
4. Unique Value 내림차순 조회와 처리값 일괄 변경
5. 날짜 분리 결과 확인: `2025년 07월 17일`, `07월17일`, `17일`, `09시`
6. Column 삭제와 CSV/XLSX 다운로드
7. Grouped bar 값 라벨, X/Y축별 정렬, Line 곡선률
8. 숫자축 범위·눈금 간격 및 범주축 범위·직접 선택

## Next Steps

`Runtime > Run all` 실행 후에도 Streamlit 서버는 계속 유지됩니다. 테스트가 끝나면 Colab의 **런타임 연결 해제 및 삭제**를 선택하면 서버도 함께 종료됩니다. 링크가 열린 뒤에는 이 노트북의 마지막까지 실행되어도 서버가 자동 종료되지 않습니다.

### 서버를 수동으로 종료하려는 경우

일반 테스트에서는 이 작업이 필요하지 않습니다. 서버만 먼저 종료하려면 새 코드 셀을 만든 뒤 다음 코드를 직접 실행하세요.

```python
cloudflared_process.terminate()
cloudflared_process.wait(timeout=10)
streamlit_process.terminate()
streamlit_process.wait(timeout=10)
```